# DiD-BCF non-main rerun: nonmain_wave_01_shard_11

This independent notebook runs 70 complete fits from these frozen study phases: diagnostic: 30, pt_impact: 30, sensitivity: 10. It uses the campaign's earlier budget: GFR 50, no additional burn-in, 500 retained draws per chain, thinning 5, and three chains. The campaign's panels, priors, and seed schedule remain pinned to the manifest.

The estimated runtime is 8.0 hours with up to two single-threaded CPU workers. Each worker is admitted only when the runtime has enough free memory; execution falls to one worker when memory or CPU limits require it. The notebook stops within 9.5 hours, including setup. Python 3.12 to 3.14 and a Colab runtime with about 12 GB RAM are appropriate. The scheduling target for this assignment is 8 hours; actual time varies by runtime.

This notebook exports paper metrics, compact diagnostics, and provenance in one ZIP. Posterior arrays, source archives, and native build files stay on the runtime disk. Verified complete fits can be resumed from a matching assignment ZIP or Drive checkpoint. An interrupted fit restarts from its original seed. Results are collected without filtering fits by diagnostics or performance.


In [ ]:
import json, os, sys, subprocess, time
from pathlib import Path
assert (3, 12) <= sys.version_info[:2] <= (3, 14), "Use Python 3.12 to 3.14"
SESSION_STARTED = time.time()
DEADLINE = SESSION_STARTED + 9.5 * 3600
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[key] = "1"
SOURCE_COMMIT = '30b165b27610dc4b2c406a9d64281d77e660257a'
REPO = Path("/content/unique_nonmain") if Path("/content").exists() else Path.cwd() / "unique_nonmain"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "https://github.com/hugogobato/DiD-BCF.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", SOURCE_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "set", 'Simulation_Studies_Revision/DiD_BCF/Nonmain_Rerun_20261014'], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == SOURCE_COMMIT, (actual, SOURCE_COMMIT)
PACKAGE = REPO / 'Simulation_Studies_Revision/DiD_BCF/Nonmain_Rerun_20261014'
manifest = json.loads((PACKAGE / "manifest.json").read_text())
requirements = [name + "==" + version for name, version in manifest["dependencies"].items()]
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                *requirements], check=True)
HELPERS = PACKAGE / "helpers"
print("Pinned non-main sources and dependencies installed:", SOURCE_COMMIT)


In [ ]:
# Optional Drive checkpoints are copied every 20 minutes.
USE_DRIVE = False
RESUME_ZIP = ""  # Set to this assignment's earlier metrics ZIP if resuming.
WORKERS = 2
NAME = 'nonmain_wave_01_shard_11'
WORK = Path("/content") / (NAME + "_work") if Path("/content").exists() else Path.cwd() / (NAME + "_work")
OUT = WORK / "nonmain_output"
OUTPUT_FILE = WORK / ("didbcf_nonmain_" + NAME + ".zip")
ASSIGNMENT = PACKAGE / "assignments" / (NAME + ".json")
DRIVE_DIR = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/DiD_BCF_Nonmain_Rerun_20261014")
    previous = DRIVE_DIR / OUTPUT_FILE.name
    if not RESUME_ZIP and previous.exists():
        RESUME_ZIP = str(previous)
# To restore a downloaded metrics ZIP, upload it in Colab's Files pane and set
# RESUME_ZIP to its /content/... path before running the preparation cell.
print("Fits:", 70, "Phases:", 'diagnostic: 30, pt_impact: 30, sensitivity: 10', "Output:", OUTPUT_FILE)


In [ ]:
command = [sys.executable, str(HELPERS / "runtime.py"), "prepare",
           "--assignment", str(ASSIGNMENT), "--work", str(WORK), "--out", str(OUT),
           "--cache", str(WORK / "source_cache"), "--deadline", str(DEADLINE)]
if RESUME_ZIP:
    command += ["--resume", RESUME_ZIP]
subprocess.run(command, check=True)


In [ ]:
command = [sys.executable, str(HELPERS / "runtime.py"), "run",
           "--assignment", str(ASSIGNMENT), "--work", str(WORK), "--out", str(OUT),
           "--setup", str(WORK / "setup_complete.json"), "--zip", str(OUTPUT_FILE),
           "--workers", str(WORKERS), "--deadline", str(DEADLINE)]
if DRIVE_DIR:
    command += ["--drive-dir", str(DRIVE_DIR)]
result = subprocess.run(command)
if result.returncode not in (0, 2):
    raise RuntimeError("Execution failed; inspect job logs and preserve the latest metrics checkpoint")
status = json.loads((OUT / "bundle" / "status.json").read_text())
print("Completed:", status["completed"], "/", status["expected"])
if status["missing_job_ids"]:
    print("Resume this same assignment with its metrics ZIP to complete",
          len(status["missing_job_ids"]), "remaining fits.")


In [ ]:
output_file = str(OUTPUT_FILE)
assert Path(output_file).is_file(), "No metrics archive was created"
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
